In [ ]:
from huggingface_hub import login

# Pega tu token aquí (empieza con "hf_...")
login(token="hf_REDACTED")

In [2]:
import pandas as pd
from datasets import load_dataset
import os

ruta_archivo = "data/sharechat_sample_balanced.parquet"

# Creamos la carpeta si no existe
os.makedirs("data", exist_ok=True)

if os.path.exists(ruta_archivo):
    print("El dataset balanceado ya existe en disco. Cargándolo...")
    df_sample = pd.read_parquet(ruta_archivo)
else:
    print("Iniciando proceso de recolección y balanceo...")
    
    # Las configuraciones exactas según la documentación de ShareChat
    plataformas = ["chatgpt", "perplexity", "grok", "gemini", "claude"]
    dataframes_crudos = []

    for plat in plataformas:
        print(f"Descargando/Cargando modelo: {plat}...")
        # Descargamos solo ese subconjunto
        ds = load_dataset("tucnguyen/ShareChat", plat, split="train")
        df_temp = ds.to_pandas()
        
        # Le inyectamos una columna para saber de qué plataforma viene 
        # (por si el dataset no la trae limpia)
        df_temp['source_platform'] = plat 
        dataframes_crudos.append(df_temp)

    # 1. Unimos todo en un mega DataFrame
    df_completo = pd.concat(dataframes_crudos, ignore_index=True)
    print(f"\nTotal de filas crudas recolectadas: {len(df_completo)}")

    # 2. Filtramos solo usuarios y los dos idiomas
    columnas_utiles = ['url', 'role', 'detected_language_final', 'plain_text', 'message_index', 'source_platform']
    df_users = df_completo[
        (df_completo['role'] == 'user') & 
        (df_completo['detected_language_final'].isin(['English', 'Spanish']))
    ][columnas_utiles]

    # 3. MUESTREO ESTRATIFICADO: Queremos balancear por Idioma y por Plataforma
    # Intentamos sacar 2000 de cada combinación. Si hay menos (ej. Claude en español), sacamos todas.
    def muestreo_inteligente(grupo, max_n=2000):
        n_disponible = len(grupo)
        n_a_extraer = min(max_n, n_disponible)
        return grupo.sample(n=n_a_extraer, random_state=42)

    print("\nAplicando muestreo estratificado...")
    df_sample = (df_users.groupby(['detected_language_final', 'source_platform'], group_keys=False)
                 .apply(muestreo_inteligente)
                 .reset_index(drop=True))

    # 4. Guardamos a disco
    df_sample.to_parquet(ruta_archivo)
    print(f"\n¡Listo! Datos guardados en: {ruta_archivo}")

# Veamos cómo quedó distribuido nuestro dataset final
print("\n--- DISTRIBUCIÓN FINAL DEL DATASET ---")
distribucion = df_sample.groupby(['detected_language_final', 'source_platform']).size().unstack(fill_value=0)
print(distribucion)
print(f"\nTotal de filas listas para etiquetar: {len(df_sample)}")

Iniciando proceso de recolección y balanceo...
Descargando/Cargando modelo: chatgpt...


chatgpt_results_final_language_filtered.(…):   0%|          | 0.00/2.29G [00:00<?, ?B/s]

c:\Users\t14\miniconda3\envs\wildchat\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\t14\.cache\huggingface\hub\datasets--tucnguyen--ShareChat. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `hf_REDACTED_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Generating train split:   0%|          | 0/976378 [00:00<?, ? examples/s]

Descargando/Cargando modelo: perplexity...


...

Descargando/Cargando modelo: claude...


claude_results_final_language_filtered.c(…):   0%|          | 0.00/283M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8364 [00:00<?, ? examples/s]


Total de filas crudas recolectadas: 1199899

Aplicando muestreo estratificado...

¡Listo! Datos guardados en: data/sharechat_sample_balanced.parquet

--- DISTRIBUCIÓN FINAL DEL DATASET ---


KeyError: 'detected_language_final'